# RQ4(3)(i): Offline Configuration Time vs Number of Vulnerabilities

Plots the cumulative offline AED/AEDI specification time as vulnerabilities are added from 21
(expert-specified) to 100 (79 generated with CVE2AP).

- Input: the 79 generated CVEs in `SAQURE-vulnerability-scalability/pddl-ap-passall/post-proposs/auto-generated/`
  and their CVE2AP generation logs in `SAQURE-vulnerability-scalability/results/offline_config/`
  (`raw_results.jsonl`, `feedback_results.jsonl`).
- Output: `experiments/plots/RQ431_SEFA.pdf` and `RQ431_SEFA.png`.

Run it from `SAQURE_Replication_Package_SEFA/`, the directory of this notebook.
`python3 create_plots.py --experiment RQ431` draws the same figure without Jupyter.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
from pathlib import Path

# Color palette (from create_plots.py)
COLORS = {
    'red': (163/255, 81/255, 57/255),
    'dark pink': (202/255, 162/255, 173/255),
    'dark orange': (234/255, 177/255, 112/255),
    'dark purple': (182/255, 175/255, 197/255),
    'dark cyan-blue': (182/255, 207/255, 212/255),
    'cyan-blue': (225/255, 241/255, 246/255),
    'purple': (230/255, 226/255, 245/255),
    'orange': (244/255, 224/255, 155/255),
    'pink': (246/255, 223/255, 224/255),
    'pink_red': (221/255, 185/255, 172/255),
}

In [ ]:
# Paths, relative to SAQURE_Replication_Package_SEFA/ (the directory of this notebook)
PKG = Path('.').resolve()
VULN_DIR = PKG / 'SAQURE-vulnerability-scalability'
AUTO_DIR = VULN_DIR / 'pddl-ap-passall' / 'post-proposs' / 'auto-generated'
RAW_PATH = VULN_DIR / 'results' / 'offline_config' / 'raw_results.jsonl'
FB_PATH  = VULN_DIR / 'results' / 'offline_config' / 'feedback_results.jsonl'
PLOTS_DIR = PKG / 'experiments' / 'plots'

# Generated, but not among the 100 CVEs (lowest CVSS among the candidates)
DROP = {'CVE-2023-34053', 'CVE-2023-34054'}
selected = set(d.name for d in AUTO_DIR.iterdir() if d.is_dir()) - DROP
print(f'Selected CVEs: {len(selected)}')

# Load records
raw_records = [json.loads(l) for l in open(RAW_PATH)]
fb_records  = [json.loads(l) for l in open(FB_PATH) if l.strip()]

# Total time per CVE
cve_total_time = {}
for cve in selected:
    raw_time = sum(r['elapsed_seconds'] for r in raw_records if r['cve_id'] == cve)
    fb_time  = sum(r['elapsed_seconds'] for r in fb_records  if r['cve_id'] == cve)
    cve_total_time[cve] = raw_time + fb_time

# Sort by time ascending, compute cumulative
sorted_cves = sorted(cve_total_time.keys(), key=lambda c: cve_total_time[c])
cumulative_times = []
cum = 0.0
for cve in sorted_cves:
    cum += cve_total_time[cve]
    cumulative_times.append(cum)

x = np.arange(1, len(sorted_cves) + 1)
y = np.array(cumulative_times)

print(f'Total: {y[-1]:.0f}s ({y[-1]/60:.1f} min), Mean: {y[-1]/79:.1f}s/CVE')

In [ ]:
FIG_SIZE = (12, 6)
FONT_SIZE_LABEL = 26
TICK_LABEL_SIZE = 22
POINTSIZE = 7
TREND_LINEWIDTH = 1.0

fig, ax = plt.subplots(figsize=FIG_SIZE)

# X axis: 21 + i for i in 1..79, so x goes from 22 to 100
x_shifted = x + 21  # x was 1..79, now 22..100
# Prepend point at 21 with y=0 (baseline: expert already done)
x_plot = np.concatenate([[21], x_shifted])
y_plot = np.concatenate([[0], y * 1000])  # convert s to ms

# Marker positions: 21, 30, 40, 50, 60, 70, 80, 90, 100
marker_x = [21, 30, 40, 50, 60, 70, 80, 90, 100]
marker_y = [np.interp(mx, x_plot, y_plot) for mx in marker_x]

# Dashed line
ax.plot(x_plot, y_plot, color=COLORS['red'], linewidth=TREND_LINEWIDTH, linestyle='--', zorder=3,
        label='Initial AED/AEDI Specification Time')

# Markers at selected positions
ax.plot(marker_x, marker_y,
        linestyle='none', marker='o', markersize=POINTSIZE,
        markeredgecolor=COLORS['red'], markerfacecolor=COLORS['red'],
        markeredgewidth=0.5, zorder=4)

# Axis labels
ax.set_xlabel('Number of Vulnerabilities', fontsize=FONT_SIZE_LABEL)
ax.set_ylabel('Configuration Time (ms)', fontsize=FONT_SIZE_LABEL)

# Y axis: 1e6 offset notation
ax.set_ylim(0, 8000000)
ax.ticklabel_format(style='sci', axis='y', scilimits=(6, 6))
ax.yaxis.get_offset_text().set_fontsize(TICK_LABEL_SIZE)

# X axis: 21 to 100
ax.set_xlim(19, 102)
ax.set_xticks([21, 30, 40, 50, 60, 70, 80, 90, 100])

# Tick settings: bottom/left inward, top none, right inward
# Left and bottom: ticks outward
ax.tick_params(axis='both', labelsize=TICK_LABEL_SIZE, which='both',
               direction='out', top=False, right=False)
ax.tick_params(axis='both', which='major', length=6, pad=4)
ax.tick_params(axis='both', which='minor', length=3)
# Right: no ticks
ax.tick_params(axis='y', which='both', right=False)

# All spines visible
for spine in ax.spines.values():
    spine.set_linewidth(1.2)

# Dense horizontal grid lines
ax.yaxis.set_minor_locator(AutoMinorLocator(5))
ax.grid(True, which='major', axis='y', linestyle='--', linewidth=0.75, alpha=0.15, color='gray')
ax.grid(True, which='minor', axis='y', linestyle='--', linewidth=0.5, alpha=0.10, color='gray')
ax.set_axisbelow(True)

ax.legend(loc='upper left', fontsize=23, frameon=True,
          edgecolor='lightgray', framealpha=0.9,
          handlelength=3, handletextpad=1.0)

fig.tight_layout(pad=1.5)

save_path = str(PLOTS_DIR / 'RQ431_SEFA.pdf')
plt.savefig(save_path, dpi=200, bbox_inches='tight')
plt.savefig(save_path.replace('.pdf', '.png'), dpi=200, bbox_inches='tight')
print(f'Saved to {save_path}')
plt.show()